# utorial

In [ ]:
import os
import json
import polars as pl
from tqdm import tqdm

import torch
from transformers import pipeline
from huggingface_hub import login

### ***1. KONFIGURACE A PŘIHLÁŠENÍ K HUGGING FACE***

In [ ]:
user_secrets = UserSecretsClient()

HF_TOKEN = user_secrets.get_secret("HF_TOKEN")

BASE_DATASET_DIR = "raw_segments_dataset"  # Root folder, where you have the comma, period, etc. folders.
OUTPUT_GOLD_DIR = "gold_datasets_output"   # Output folder for finished JSONs

### ***Map of all 5 of your Parquet datasets***

In [ ]:
DATASET_MAP = {
    "comma": os.path.join(BASE_DATASET_DIR, "comma", "part-0.parquet"),
    "exclamation_mark": os.path.join(BASE_DATASET_DIR, "exclamation_mark", "part-0.parquet"),
    "none": os.path.join(BASE_DATASET_DIR, "none", "part-0.parquet"),
    "period": os.path.join(BASE_DATASET_DIR, "period", "part-0.parquet"),
    "question_mark": os.path.join(BASE_DATASET_DIR, "question_mark", "part-0.parquet")
}

### ***Converter of internal positional tags of the Czert model to your Czech names of parts of speech***

In [ ]:
def map_czert_tag_to_czech(tag):
    if not tag:
        return "jiny"
    
    # The model sometimes returns B- or I- prefixes due to tokenization, we will cut those off
    clean_tag = tag.split("-")[-1] if "-" in tag else tag
    
    # Czert uses the official Prague Corpus: the first letter determines the basic part of speech
    char_0 = clean_tag[0] if len(clean_tag) > 0 else ""
    
    if char_0 == "N": return "podstatne_jmeno"
    elif char_0 == "V": return "sloveso"
    elif char_0 in ["P", "A"]: return "zajmeno"  # Includes pronouns and determiners
    elif char_0 == "D": return "prislovce"
    elif char_0 == "R": return "predlozka"
    return "jiny"

## ***2. PIPELINE CORE (Processing one complete file)***

In [ ]:
def process_single_dataset(punc_key, parquet_path, nlp_pipeline):
    if not os.path.exists(parquet_path):
        print(f"⚠️ Warning: File for '{punc_key}' not found in path: {parquet_path}. Skipping.")
        return

    print(f"\n Loading COMPLETE dataset [{punc_key}] from: {parquet_path}")
    df = pl.read_parquet(parquet_path)

    # Load 100% of the data from Parquet, filter out empty rows, and convert to dicts
    raw_data = (
        df.select(["segment", "punctuation_type"])
        .filter(pl.col("segment").is_not_null())
        .to_dicts()
    )

    gold_dataset = []
    
   # The loop runs through all the lines in the file
    for row in tqdm(raw_data, desc=f"⏳ Czert Tagging -> {punc_key}"):
        segment_text = row["segment"]
        punc_type = row["punctuation_type"]
        
        try:
            # The model analyzes the entire sentence
            predictions = nlp_pipeline(segment_text)
            
            tokens_annotation = []
            has_subject = False
            
            for pred in predictions:
                word = pred["word"].strip()
                
                # Clean up special characters that BERT models add before words/subwords
                word = word.replace("Ġ", "").replace("##", "")
                
                # We don't want to store punctuation and empty tokens in word annotations
                if not word or word in [".", ",", "!", "?", "\"", "“", "”"]:
                    continue
                    
                raw_tag = pred["entity"] if "entity" in pred else pred["entity_group"]
                slovni_druh = map_czert_tag_to_czech(raw_tag)
                
                # Deterministic syntax rules (replace unreliable LLM)
                vetny_clen = "jiny"
                pad = 0
                
                if slovni_druh == "sloveso":
                    vetny_clen = "prisudek"
                elif slovni_druh == "podstatne_jmeno" and not has_subject:
                    vetny_clen = "podmet"
                    pad = 1  # The subject in Czech represents the 1st case
                    has_subject = True
                elif slovni_druh == "podstatne_jmeno":
                    pad = 4  # We default other nouns to the 4th case (who/what)
                
                # Inflexible words have no fall at all
                if slovni_druh in ["predlozka", "prislovce"]:
                    pad = 0

                tokens_annotation.append({
                    "slovo": word,
                    "slovni_druh": slovni_druh,
                    "vetny_clen": vetny_clen,
                    "pad": pad
                })
                
            gold_dataset.append({
                "segment": segment_text,
                "punctuation_type": punc_type,
                "tokens_annotation": tokens_annotation
            })

        except Exception:
            # If the tokenizer fails on a non-standard character, we safely jump to the next sentence
            continue

    # Save the resulting complete JSON for the given punctuation type
    output_path = os.path.join(OUTPUT_GOLD_DIR, f"gold_dataset_{punc_key}.json")
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(gold_dataset, f, ensure_ascii=False, indent=2)
        
    print(f"💾 Dataset [{punc_key}] completely processed and saved to: {output_path} (Sentence: {len(gold_dataset)})")

## ***3. MAIN STARTING CYCLE***

In [ ]:
if __name__ == "__main__":
    print("🔬 Initializing Local Research Pipeline (UWB CZERT - FULL MODE)")
    print("---------------------------------------------------------------------------")
    
    # 1. Login to Hugging Face Hub via token
    print("🔑 Authentication on Hugging Face Hub...")
    login(token=HF_TOKEN)
    print("🔓 Login successful.")
    
    # Create output folder if it doesn't exist yet
    os.makedirs(OUTPUT_GOLD_DIR, exist_ok=True)
    
    # 2. Hardware detection (if you have local CUDA graphics, it will run on it, otherwise CPU)
    device = 0 if torch.cuda.is_available() else -1
    print(f"⏳ Downloading/Loading official model 'UWB-AIR/Czert-B-base-cased'...")
    
    # Initialize the Hugging Face pipeline
    nlp_pipeline = pipeline(
        "token-classification",
        model="UWB-AIR/Czert-B-base-cased",
        aggregation_strategy="simple",  # Automatically joins BERT subwords into whole words
        device=device,
        token=HF_TOKEN
    )
    print(f" Model is ready and uploaded (Running on: {'GPU' if device == 0 else 'CPU'}).")

    # 3. Start processing for all 5 files sequentially one after the other
    for punc_key, parquet_path in DATASET_MAP.items():
        process_single_dataset(punc_key, parquet_path, nlp_pipeline)
        
    print("\n---------------------------------------------------------------------------")
    print(f"🎉 DONE! All Parquet files have been completely transformed.")
    print(f"You can find the resulting error-free structures in the folder: '{OUTPUT_GOLD_DIR}'")